# GLM3微调代码理解笔记

GLM3创新点：使用均方根rms归一化，相对于ChatGLM，词典变小（相同：Rotary旋转位置编码，embedding，28层多头注意力，MLP，LN层）

GLM4创新点：相对于ChatGLM3，词典变大适应更多语言；使用40层layer，分组查询注意力（Grouped-Query Attention, GQA）/ 多查询注意力（MQA）的变体

## 1. 代码结构概述

GLM3微调代码主要基于Hugging Face的Transformers库和PEFT（Parameter-Efficient Fine-Tuning）库，实现了对GLM3模型的微调。代码结构清晰，功能模块化，易于扩展和维护。主要包括以下几个模块：

- 数据处理：加载数据集、预处理数据、构建数据管理器。
- 模型加载：加载预训练模型和分词器，支持PEFT配置（如LoRA、Prefix-Tuning）。
- 训练与评估：使用Seq2SeqTrainer进行模型训练和评估，支持自动从检查点恢复训练。
- 工具函数：包括数据整理、模型保存、指标计算等。

## 2. 核心功能解析

### (1) 数据处理
- 数据加载：使用datasets库加载数据集，支持多种格式（如.csv、.json、.jsonl）。
- 数据预处理：将对话数据转换为模型输入格式（input_ids和labels），用于训练。生成评估用的输入数据（input_ids和output_ids），-不计算损失。
- 数据整理：使用DataCollatorForSeq2Seq对输入数据进行填充（padding），确保批次数据长度一致。

### (2) 模型加载
- 加载分词器和模型：使用AutoTokenizer和AutoModelForCausalLM加载预训练模型和分词器。支持PEFT配置，通过get_peft_model将模型转换为参数高效微调模式。
- 模型规模打印：print_model_size函数用于打印模型的可训练参数量。

### (3) 训练与评估
- 训练配置：从YAML文件中加载微调配置（FinetuningConfig），包括数据路径、模型超参数、训练参数等。
- 训练器：使用Seq2SeqTrainer进行模型训练和评估，支持从检查点自动恢复训练。
- 评估指标：计算ROUGE和BLEU指标，用于评估生成文本的质量。

### (4) 工具函数
- 路径解析：_resolve_path函数解析文件路径，支持相对路径和用户目录（~）。
- 配置文件解析：_get_yaml_parser函数加载和解析YAML配置文件。
- 模型准备：_prepare_model_for_training函数将模型参数转换为fp32格式，确保训练稳定性。

## 3. 主要流程

- 加载配置：从YAML文件中加载微调配置（FinetuningConfig）。
- 加载模型和分词器：根据配置加载预训练模型和分词器，支持PEFT配置。
- 加载和预处理数据：使用DataManager加载数据集，并通过process_batch和process_batch_eval预处理数据。
- 训练模型：使用Seq2SeqTrainer进行模型训练，支持从检查点恢复训练。
- 评估模型：在验证集和测试集上评估模型性能，计算ROUGE和BLEU指标。

## 4. 关键代码片段解析

### (1) 数据预处理

In [ ]:
def load_tokenizer_and_model(model_dir, peft_config=None):
    '''
    加载分词器和模型

    Args:
        model_dir (str): 模型目录路径
        peft_config (Optional[PeftConfig]): PEFT配置对象,用于参数高效微调。默认为None

    Returns:
        tuple: 包含以下元素的元组:
            - tokenizer (AutoTokenizer): 加载的分词器
            - model (AutoModelForCausalLM): 加载的模型
    '''
    # 从指定目录加载分词器,trust_remote_code=True表示信任并允许执行模型中的自定义代码
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    
    # 如果提供了peft配置
    if peft_config is not None:
        # 首先加载原始的因果语言模型
        model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=True)
        # 使用peft配置将模型转换为参数高效微调(PEFT)模式
        model = get_peft_model(model, peft_config)
    else:
        # 如果没有peft配置,直接加载原始的因果语言模型
        model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=True)
    
    # 返回加载的分词器和模型
    return tokenizer, model


- 将对话数据转换为模型输入格式，生成input_ids和labels。

### (2) 模型加载

In [ ]:
def load_tokenizer_and_model(model_dir, peft_config=None):
    '''
    加载分词器和模型

    Args:
        model_dir (str): 模型目录路径
        peft_config (Optional[PeftConfig]): PEFT配置对象,用于参数高效微调。默认为None

    Returns:
        tuple: 包含以下元素的元组:
            - tokenizer (AutoTokenizer): 加载的分词器
            - model (AutoModelForCausalLM): 加载的模型
    '''
    # 从指定目录加载分词器,trust_remote_code=True表示信任并允许执行模型中的自定义代码
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    
    # 判断是否提供了peft配置对象
    if peft_config is not None:
        # 如果提供了peft配置,首先加载原始的因果语言模型
        model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=True)
        # 使用peft配置将模型转换为参数高效微调(PEFT)模式
        model = get_peft_model(model, peft_config)
    else:
        # 如果没有peft配置,直接加载原始的因果语言模型
        model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=True)
    
    # 返回加载的分词器和模型作为元组
    return tokenizer, model


- 加载预训练模型和分词器，支持PEFT配置。

### (3) 训练与评估

In [ ]:
# 创建Seq2SeqTrainer对象,用于序列到序列的训练任务
trainer = Seq2SeqTrainer(
    model=model,  # 传入要训练的模型
    args=ft_config.training_args,  # 传入训练参数配置
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),  # 使用Seq2Seq数据整理器,用于将样本批量处理成模型输入格式
    train_dataset=train_dataset,  # 传入训练数据集
    eval_dataset=val_dataset,  # 传入验证数据集
    tokenizer=tokenizer,  # 传入分词器
    compute_metrics=functools.partial(compute_metrics, tokenizer=tokenizer),  # 设置评估指标计算函数,并固定tokenizer参数
)

# 开始训练过程
trainer.train()


- 使用Seq2SeqTrainer进行模型训练和评估。

## 5. 超参数设置


### 全量微调

- 模型参数：全部模型参数都会在微调过程中更新。
- 学习率：通常需要较小的学习率以避免破坏预训练模型的知识。
- 批次大小：根据GPU/TPU内存大小调整。
- 训练步数：确定训练的总时长。
- 优化器：选择Adam、AdamW等不同的优化器。

### LoRA微调

- 模型参数：只更新模型的特定部分（通常是自注意力机制的权重），其他部分参数保持不变。
- LoRA特定参数：
  - peft_type：指定使用的PEFT类型，这里是LoRA。
  - r：LoRA中的秩（rank），控制微调参数的数量。
  - lora_alpha：LoRA中用于调整权重的alpha参数。
  - lora_dropout：LoRA中使用的dropout比率。
- 学习率：由于LoRA更新的参数较少，可以使用相对较高的学习率。
- 批次大小：由于LoRA计算量较小，可能允许使用较大的批次大小。
- 训练步数：可能需要更多的训练步数来达到与全量微调相同的性能水平。

## 6. 微调后的评估

评估指标为中文Rouge score（罗格分数）和BLEU-4。生成的结果保存在./output/adgen-chatglm-6b-pt-8-1e-2/generated_predictions.txt，评估指标保存在predict_results.json。

## 7. 微调后的推理

推理时需要修改cli_demo.py中的模型名字，改为微调后的检查点路径：

In [ ]:
MODEL_PATH = os.environ.get('MODEL_PATH', '/root/GLM3 整合包/微调/lora微调/output/checkpoint-500') # 获取模型路径修改checkpoint-500

然后执行推理脚本：

In [ ]:
bash /root/chuli/微调/lora/lora推理.sh

## 8. 微调存在的问题

微调后仍存在遗忘问题，即增加了某方面的能力，但原有能力受到影响。为保持原有能力，同时具备新能力，需要多次全量训练，重新学习分布。

## 9. 标签平滑（Label Smoothing）

标签平滑通过对标签分布进行平滑处理，缓解模型对训练数据中标签的过度自信或过度拟合，提高模型的泛化能力。

## 10. Top-k与Top-p采样

- Top-k采样：从概率分布中选择前k个最有可能的候选词。
- Top-p采样：选择一组候选词，使这些词的累计概率达到或超过p（0 < p ≤ 1）。

## 11. 微调步骤

### 1.进入虚拟环境：

In [ ]:
conda init
source ~/.bashrc
conda activate py310_chat

### 2.数据预处理：

In [ ]:
python /root/chuli/微调/一条龙.py 

### 3.微调训练：

In [ ]:
bash /root/chuli/微调/lora/lora训练.sh

# chattglm4官方模型微调
bash /root/chuli/微调/lora微调/lora训练.sh

### 4.微调后的权重及评估指标路径：
/root/GLM3 整合包/微调/lora微调/output/checkpoint-500
/rootchat_robot_glm4_lora.py

### 5.微调推理/聊天机器人：

In [ ]:
bash /root/chuli/微调/lora/lora推理.sh

# chattglm4上线，先管道映射：ssh-CNg
python chat_robot_glm4_lora.py

## 12. 改进建议

- 增加日志记录：添加日志记录功能，方便调试和监控训练过程。
- 支持更多PEFT方法：扩展支持更多PEFT方法。
- 优化数据预处理：对于大规模数据集，进一步优化数据预处理流程，提高效率。

## 13. 总结

这段代码实现了一个完整的序列到序列模型微调框架，支持多种数据格式、参数高效微调和自动恢复训练。适用于对话生成、文本摘要等任务的微调场景。通过模块化设计和PEFT支持，代码在资源有限的环境中表现出色，并且易于扩展和维护。